In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv(r"C:\Users\Matthew .K. Maunga\OneDrive\Desktop\MSU project\MSU_Lost_Found_ML_System\data\raw\descriptions.csv")

BASE = r"C:\Users\Matthew .K. Maunga\OneDrive\Desktop\MSU project\MSU_Lost_Found_ML_System"

print(f"TensorFlow version: {tf.__version__}")
print(f" Dataset loaded: {len(df)} records")
print(f" Libraries imported successfully")

TensorFlow version: 2.21.0
 Dataset loaded: 172 records
 Libraries imported successfully


In [2]:
# Load MobileNetV2 without top classification layer
print("Loading MobileNetV2 model...")
model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
print(f" Model loaded!")
print(f"   Input shape  : {model.input_shape}")
print(f"   Output shape : {model.output_shape}")

# Image preprocessing function
def preprocess_image(img_path):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)
        return img_array
    except Exception as e:
        print(f"  Error loading {img_path}: {e}")
        return None

# Extract features from all images
print("\nExtracting features from all 172 images...")
features = []
valid_indices = []

for idx, row in df.iterrows():
    img_path = os.path.join(BASE, row['image_path'])
    img_array = preprocess_image(img_path)
    
    if img_array is not None:
        feature = model.predict(img_array, verbose=0)
        features.append(feature[0])
        valid_indices.append(idx)
    else:
        print(f"  Skipping: {row['image_path']}")

features = np.array(features)
print(f"\n Features extracted!")
print(f"   Shape: {features.shape}")
print(f"   Valid images: {len(valid_indices)} out of {len(df)}")

Loading MobileNetV2 model...


C:\Users\Matthew .K. Maunga\AppData\Local\Temp\ipykernel_12144\3188137769.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


 Model loaded!
   Input shape  : (None, None, None, 3)
   Output shape : (None, 1280)

Extracting features from all 172 images...

 Features extracted!
   Shape: (172, 1280)
   Valid images: 172 out of 172


In [3]:
# Save image features
PROCESSED_PATH = r"C:\Users\Matthew .K. Maunga\OneDrive\Desktop\MSU project\MSU_Lost_Found_ML_System\data\processed"

with open(f"{PROCESSED_PATH}\\image_features.pkl", "wb") as f:
    pickle.dump({
        'features': features,
        'item_ids': df['item_id'].tolist(),
        'categories': df['category'].tolist(),
        'status': df['status'].tolist(),
        'image_paths': df['image_path'].tolist()
    }, f)

print(" Image features saved to data/processed/image_features.pkl")

# Test image similarity
lost_mask  = df['status'] == 'lost'
found_mask = df['status'] == 'found'

lost_features  = features[lost_mask]
found_features = features[found_mask]
lost_items     = df[lost_mask].reset_index(drop=True)
found_items    = df[found_mask].reset_index(drop=True)

# Compare first lost item against all found items
scores = cosine_similarity([lost_features[0]], found_features)[0]
best_match_idx = scores.argmax()

print(f"\n--- Test Image Match ---")
print(f"Lost item     : {lost_items['item_name'].iloc[0]} ({lost_items['category'].iloc[0]})")
print(f"Best match    : {found_items['item_name'].iloc[best_match_idx]} ({found_items['category'].iloc[best_match_idx]})")
print(f"CV Score      : {scores[best_match_idx]:.4f}")

 Image features saved to data/processed/image_features.pkl

--- Test Image Match ---
Lost item     : Backpack (backpack)
Best match    : Backpack (backpack)
CV Score      : 1.0000
